# TN3 — Quét toàn dải alpha, DS-TCN 192 kênh tầm nhìn 121

## Chỗ lệch mà thí nghiệm này khép lại

Cả đồ án **train bằng MSE** nhưng **chấm bằng Pearson**. Pearson bất biến với
thang đo — chỉ quan tâm hình dạng. MSE thì ngược lại:

    dự báo A: đúng hình dạng, biên độ gấp 3   MSE 2,0000   Pearson +1,0000
    dự báo B: phẳng lì, đoán bừa số 0         MSE 0,5000   Pearson  0,0000

MSE chọn B, Pearson chọn A.

```
loss = alpha × MSE + (1 − alpha) × (1 − Pearson)
```

## Quét gì

**Mười mức alpha, từ 0,0 tới 0,9**, mỗi mức đủ bốn fold, một seed.

`alpha = 1,0` **không chạy** — đó chính là MSE thuần, đã có kết quả
**0,764428 (1 seed)**. Ô so sánh ở mục 5 tự chèn nó vào làm điểm tham chiếu.

    alpha 1,0   MSE thuần        <- đã có, không chạy lại
    alpha 0,9 .. 0,1             <- quét
    alpha 0,0   Pearson thuần

Quét đủ mười một điểm thì thấy được **hình dạng đường cong**, và biết đỉnh nằm
ở đâu chứ không giả định từ khảo sát trên bộ mã khác.

## Hai đầu mút đáng chú ý

**alpha = 1,0 (MSE thuần)** là chỗ lệch hoàn toàn: train một tiêu chí, chấm
tiêu chí khác.

**alpha = 0,0 (Pearson thuần)** khớp hoàn hảo với cách chấm, nhưng có bẫy số
học. Mẫu số Pearson là `‖pred‖·‖target‖ + 1e-8`. Dự báo càng phẳng, gradient
càng lớn — đo được trên chính hàm loss này:

| biên độ dự báo | gradient lớn nhất |
|---:|---:|
| 1e-2 | 4,6 |
| 1e-4 | 813 |
| 1e-6 | 83.000 |
| 1e-8 | **99.800.000** |

Không thành `NaN` nhờ `+1e-8`, nhưng một bước Adam với gradient cỡ đó đủ làm
hỏng trọng số. Phần MSE trong hàm lai chính là thứ kéo model ra khỏi vùng
phẳng. Nếu `alpha = 0,0` cho kết quả kỳ lạ thì xem `train_mse` ở đầu ra.

## Cấu hình nền

| | |
|---|---|
| model | `ds_tcn --channels 192 --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element` |
| tham số | **310.873** |
| tầm nhìn | 121 |
| MSE thuần, alpha 1,0 | **0,764428** *(1 seed)* |

## Thời gian và cách chạy từng phần

Mười mức × bốn fold ≈ **8,7 giờ**. Mỗi alpha một ô riêng, chạy độc lập.

**Không cần chạy hết một lần.** Dừng lúc nào cũng được; mở lại, chạy ô khôi
phục ở mục 1 rồi bấm tiếp các ô còn thiếu. `run_cv.py` bỏ qua mọi fold đã xong.

Muốn thấy hình dạng sớm thì chạy thưa trước: **0,0 → 0,5 → 0,9 → 0,7 → 0,3**,
rồi lấp các điểm còn lại sau.

## Đọc kết quả

Nền có 1 seed, các mức alpha mới chỉ **một seed**. `seed_std` của tám cấu hình
TN1 trải từ 0,0007 tới 0,0108 — chênh dưới khoảng 0,01 chưa đọc được gì.

Nhưng quét mười một điểm thì **hình dạng đường cong** đáng tin hơn từng điểm
riêng lẻ. Có đỉnh rõ và hai bên dốc xuống thì đó là tín hiệu thật.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn. Hậu tố `alpha` trong tên cấu hình chỉ có ở bản mới.

In [ ]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

Khôi phục mọi mức alpha đã chạy.

**Chạy ô này mỗi khi mở lại notebook.** Nó gộp `runs/*/summary.csv` vào
`runs/summary.csv` — chỗ `run_cv.py` tra để biết fold nào đã xong. Nhờ đó bấm
lại ô alpha cũ chỉ mất vài giây thay vì train lại.

In [ ]:
import glob, subprocess, os, csv
for f in sorted(glob.glob("/content/drive/MyDrive/mobivital/tn3_*c192*.zip")):
    subprocess.run(["unzip", "-oq", f, "-d", "runs/"], check=True)
# run_cv.py tra runs/summary.csv, mà tệp nén chỉ có runs/<thí nghiệm>/summary.csv
rows, seen = [], set()
for s in glob.glob("runs/*/summary.csv"):
    for r in csv.DictReader(open(s)):
        k = (r.get("experiment"), r.get("run_id"))
        if k not in seen: seen.add(k); rows.append(r)
if rows:
    w = csv.DictWriter(open("runs/summary.csv", "w", newline=""), fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

## 2. Kiểm bản cài đặt

Số tham số phải ra đúng **310.873**.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. Mười mức alpha, mỗi mức đủ 4 fold

Tên cấu hình: `ds_tcn_c192_k5_n4_none_do0.2_dpel_mse_pearson_a<A>_corr0.9_seed0`.

Mỗi ô khoảng **52 phút**. Chạy được rời rạc, không cần theo thứ tự.

**alpha 0,0 — MSE 0 phần trăm, Pearson 100 phần trăm**  — **Pearson thuần**, khớp hoàn toàn với cách chấm điểm

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 0

**alpha 0,1 — MSE 10 phần trăm, Pearson 90 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.1 --seed 0

**alpha 0,2 — MSE 20 phần trăm, Pearson 80 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.2 --seed 0

**alpha 0,3 — MSE 30 phần trăm, Pearson 70 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.3 --seed 0

**alpha 0,4 — MSE 40 phần trăm, Pearson 60 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.4 --seed 0

**alpha 0,5 — MSE 50 phần trăm, Pearson 50 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.5 --seed 0

**alpha 0,6 — MSE 60 phần trăm, Pearson 40 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.6 --seed 0

**alpha 0,7 — MSE 70 phần trăm, Pearson 30 phần trăm**  — đỉnh của khảo sát trên bộ mã khác

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.7 --seed 0

**alpha 0,8 — MSE 80 phần trăm, Pearson 20 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.8 --seed 0

**alpha 0,9 — MSE 90 phần trăm, Pearson 10 phần trăm**

In [ ]:
!python scripts/run_cv.py --experiment tn3 --model ds_tcn --channels 192 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.9 --seed 0

## 4. Cất kết quả

Chạy ô này sau mỗi đợt, đừng đợi xong hết.

In [ ]:
!python scripts/save_results.py tn3 --out tn3_ds_tcn_c192

## 5. Bảng alpha, kèm điểm tham chiếu

`alpha = 1,0` không nằm trong `runs/tn3/` vì nó chạy dưới tên `--loss mse` ở
nhóm `tn1`. Ô dưới chèn nó vào làm dòng tham chiếu.

In [ ]:
import csv, re, os
MSE_THUAN = 0.764428
d = {}
if os.path.exists("runs/tn3/summary.csv"):
    d = {float(re.search(r"_a([\d.]+)_", r["run_id"]).group(1)): float(r["score_macro"])
         for r in csv.DictReader(open("runs/tn3/summary.csv"))
         if r["fold"] == "TONG" and "_c192_" in r["run_id"]}
d[1.0] = MSE_THUAN
for a in sorted(d):
    print("  alpha", a, " ", round(d[a], 6), "<- MSE thuần" if a == 1.0 else "")

## 6. Đường cong alpha

Chấm đỏ là MSE thuần, lấy từ kết quả đã có.

In [ ]:
import matplotlib.pyplot as plt
x = sorted(d)
plt.plot(x, [d[i] for i in x], "o-", label="quét alpha")
plt.plot([1.0], [MSE_THUAN], "ro", ms=10, label="MSE thuần (đã có)")
plt.xlabel("alpha — trọng số MSE"); plt.ylabel("cv_score")
plt.legend(); plt.grid(alpha=.3); plt.title("0 = Pearson thuần, 1 = MSE thuần")

## 7. Ngắt phiên

Chạy dở cũng ngắt được — kết quả đã nén sang Drive sau mỗi fold.

In [ ]:
from google.colab import runtime
runtime.unassign()